# OCR toàn bộ Cap_01 và Cap_02

Chỉ trang **02, 06, 48** của **mỗi cặp** có thư mục `raw/` theo kiểu notebook cũ: ảnh đầu vào PNG, ảnh `_ocr_res_img.png` có chữ bên cạnh, JSON `rec_texts`/`rec_scores`, và `result.txt`. Những trang còn lại chỉ xuất kết quả trang như notebook mới, không lưu ảnh crop/raw.


## A. Cấu hình đường dẫn và các phụ thuộc (chạy một lần)

In [1]:
from pathlib import Path
import os, sys, json, time, gc, math, subprocess, importlib.util, warnings

ROOT = Path('/kaggle/input/datasets/anhtuanxxx/nlp-midterm-phu-bien-tap-luc/01_nguon')
CAPS = ['Cap_01', 'Cap_02']
PDF_PATHS = {cap: ROOT / cap / 'HAN.pdf' for cap in CAPS}
from tempfile import mkdtemp
OUTPUT_ROOT = Path(mkdtemp(prefix='HAN_OCR_full_', dir='/kaggle/working'))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DPI = 144
MAX_SIDE = 2000
CROP_HEIGHT_RATIO = 0.97
PAD_FOR_OCR = 25
EXPECTED_TEXT_COLUMNS = 16    # Chỉ để cảnh báo, KHÔNG tự xóa để ép số cột
SPACED_COLUMN_CAPS = {'Cap_02'}
DESKEW_CAPS = {'Cap_01'}
MAX_ANGLE, COARSE_STEP, FINE_STEP = 5.0, 0.25, 0.05
MIN_ANGLE, MIN_GAIN = 0.10, 0.002
PAGE_ROTATIONS = {('Cap_01', 6):2.0, ('Cap_01',7):0.5,('Cap_01',48):-1.0}
USE_MANUAL_OVERRIDE = False

BORDER_MIN_SCORE = 2.4

# Kích thước ảnh preview / trang: 16-20 cột thành 1 ảnh JPEG, ~1100×340 px
THUMB_WIDTH = 57
THUMB_HEIGHT = 340
PREVIEW_JPEG_QUALITY = 68
SHOW_PREVIEW = False
RUN_OCR_BY_DEFAULT = False

for cap,p in PDF_PATHS.items():
    print(cap, p, 'FOUND' if p.is_file() else 'MISSING')
    if not p.is_file(): raise FileNotFoundError(p)

packages = {'cv2':'opencv-python-headless','numpy':'numpy','scipy':'scipy',
            'pymupdf':'pymupdf','PIL':'pillow','pandas':'pandas',
            'xlsxwriter':'xlsxwriter','tqdm':'tqdm','IPython':'ipython'}
missing = [pkg for module,pkg in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing:', missing)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
else:
    print('Core dependencies OK')

# Thông tin GPU để kiểm tra sớm; preview hình học vẫn chạy được trên CPU.
try:
    probe = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
    print('GPU:',probe.stdout.strip() if probe.returncode==0 else 'Not detected')
except (FileNotFoundError, subprocess.TimeoutExpired):
    print('GPU: Not detected (preview geometry still available)')

Cap_01 /kaggle/input/datasets/anhtuanxxx/nlp-midterm-phu-bien-tap-luc/01_nguon/Cap_01/HAN.pdf FOUND
Cap_02 /kaggle/input/datasets/anhtuanxxx/nlp-midterm-phu-bien-tap-luc/01_nguon/Cap_02/HAN.pdf FOUND
Core dependencies OK
GPU: Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## D. Thiết lập PaddleOCR GPU (chỉ chạy trước khi `do_ocr=True`)

In [2]:
def setup_ocr():
    global ocr_han
    if 'ocr_han' in globals():
        return ocr_han
    # ModelScope imports Torch when it detects the optional package.
    import importlib
    if importlib.util.find_spec('torch') is not None:
        from importlib.metadata import distributions
        installed = {d.metadata.get('Name', '').lower() for d in distributions() if d.metadata.get('Name')}
        packages = [p for p in ('torchvision', 'torchaudio', 'torch') if p in installed]
        if packages:
            print('Gỡ bộ Torch trong môi trường Kaggle dành cho PaddleOCR...', flush=True)
            subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', *packages])
        importlib.invalidate_caches()
        if 'torch' in sys.modules:
            raise RuntimeError('Torch đã được nạp. Restart kernel rồi chạy Run All lại.')
        # A leftover directory without __init__.py is a namespace package.
        # Rename only this specific installed namespace; retain its contents.
        from pathlib import Path
        from uuid import uuid4
        import site, shutil
        spec = importlib.util.find_spec('torch')
        if spec is not None and spec.origin is None:
            roots = {Path(root).resolve() for root in site.getsitepackages()}
            for entry in list(spec.submodule_search_locations or []):
                folder = Path(entry)
                if (folder.name == 'torch' and folder.parent.resolve() in roots
                        and not folder.is_symlink() and not (folder / '__init__.py').exists()):
                    shutil.move(str(folder), str(folder.with_name('torch_uninstalled_' + uuid4().hex)))
            importlib.invalidate_caches()
        if importlib.util.find_spec('torch') is not None:
            raise RuntimeError('pip đã gỡ Torch nhưng kernel vẫn tìm thấy torch. Kiểm tra đường dẫn: '
                               + str(importlib.util.find_spec('torch')))
    if importlib.util.find_spec('paddle') is None:
        # Trùng nguồn wheel của notebook gốc. Nếu Kaggle đổi CUDA, cần điều chỉnh index.
        subprocess.check_call([sys.executable,'-m','pip','install','-q',
            'paddlepaddle-gpu==3.2.0','-i',
            'https://www.paddlepaddle.org.cn/packages/stable/cu126/'])
    if importlib.util.find_spec('paddleocr') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','paddleocr'])
    import paddle
    from paddleocr import PaddleOCR
    print('Paddle:',paddle.__version__,'GPU count:',paddle.device.cuda.device_count())
    if paddle.device.cuda.device_count()<1:
        raise RuntimeError('Chưa có Paddle GPU: hãy bật Accelerator GPU, kiểm tra CUDA và khởi động lại session nếu vừa cài.')
    paddle.set_device('gpu:0')
    with paddle.no_grad():
        probe = paddle.nn.Conv2D(3, 4, 3)(paddle.randn([1, 3, 32, 32]))
        probe.numpy()  # Đồng bộ để phát hiện lỗi GPU trước khi OCR.
    print('GPU Conv2D OK', flush=True)
    ocr_han=PaddleOCR(
        lang='chinese_cht',device='gpu:0',
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,use_textline_orientation=True,
        text_det_limit_side_len=2000,text_det_limit_type='max',
        text_det_box_thresh=.3,text_det_thresh=.2,
        text_det_unclip_ratio=2.0,text_rec_score_thresh=0.0)
    return ocr_han

## B. Deskew và tách cột. Mọi trang Cap_02 dùng khoảng cách cột theo từng nửa trang.

In [3]:
import cv2, numpy as np, pymupdf, pandas as pd
from scipy.ndimage import median_filter
from PIL import Image, ImageDraw, ImageFont
from io import BytesIO
from IPython.display import display, Image as IPyImage
from tqdm.auto import tqdm

def rotate_image_bound(img, angle):
    if angle == 0:
        return img

    h, w = img.shape[:2]
    cx, cy = w / 2, h / 2

    M = cv2.getRotationMatrix2D(
        (cx, cy), angle, 1.0
    )

    cos = abs(M[0, 0])
    sin = abs(M[0, 1])

    new_w = int(np.ceil(w * cos + h * sin))
    new_h = int(np.ceil(h * cos + w * sin))

    M[0, 2] += new_w / 2 - cx
    M[1, 2] += new_h / 2 - cy

    return cv2.warpAffine(
        img,
        M,
        (new_w, new_h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(255, 255, 255)
    )

def preprocess_page(
    img,
    max_side=MAX_SIDE,
    rotate_deg=None
):
    if img is None or img.size == 0:
        return None
    if rotate_deg is None:
        rotate_deg = 0.0

    # Crop 3% chiều cao phía dưới
    h, w = img.shape[:2]

    img = img[:int(h * CROP_HEIGHT_RATIO), :]

    # Xoay riêng từng trang
    img = rotate_image_bound(img, rotate_deg)

    # Grayscale
    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    # Denoising
    gray = cv2.fastNlMeansDenoising(
        gray,
        None,
        h=10,
        templateWindowSize=7,
        searchWindowSize=21
    )

    # Contrast enhancement
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    gray = clahe.apply(gray)

    # Resize
    h_res, w_res = gray.shape

    s = max_side / max(h_res, w_res)

    if s != 1:
        gray = cv2.resize(
            gray,
            (
                int(w_res * s),
                int(h_res * s)
            ),
            interpolation=(
                cv2.INTER_AREA
                if s < 1
                else cv2.INTER_CUBIC
            )
        )

    return gray

def detect_column_ranges(gray_img, cap):

    _, thresh = cv2.threshold(
        gray_img,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    vertical_projection = np.sum(
        thresh,
        axis=0
    )

    smoothed_proj = median_filter(
        vertical_projection,
        size=9
    )

    threshold = np.max(smoothed_proj) * 0.10

    is_gap = smoothed_proj < threshold

    h, w = gray_img.shape

    # Cap_02 thường KHÔNG có dải gáy đen; không ép gap giả lên chữ.
    # Giữ nguyên logic gốc cho Cap_01.
    if cap == "Cap_01":
        # Phát hiện vùng gáy sách
        mid_start = int(w * 0.40)
        mid_end = int(w * 0.60)

        if mid_end > mid_start:

            spine_x = (
                mid_start
                + np.argmax(
                    smoothed_proj[mid_start:mid_end]
                )
            )

            gap_radius = 20

            is_gap[
                max(0, spine_x - gap_radius):
                min(w, spine_x + gap_radius)
            ] = True

    ranges = []
    in_column = False
    start_x = 0

    for x in range(len(is_gap)):

        if not is_gap[x] and not in_column:

            start_x = x
            in_column = True

        elif is_gap[x] and in_column:

            end_x = x
            in_column = False

            width = end_x - start_x

            if width <= 20:
                continue

            # Tách đôi nếu cột quá rộng
            if width > 150:

                mid_c = start_x + width // 2
                search_range = 40

                s_start = max(
                    start_x,
                    mid_c - search_range
                )

                s_end = min(
                    end_x,
                    mid_c + search_range
                )

                if s_end > s_start:

                    split_x = (
                        s_start
                        + np.argmin(
                            smoothed_proj[s_start:s_end]
                        )
                    )

                    ranges.extend([
                        (start_x, split_x),
                        (split_x, end_x)
                    ])

                    continue

            ranges.append(
                (start_x, end_x)
            )

    # Cột cuối cùng
    if in_column:
        ranges.append((start_x, w))

    # Thứ tự đọc chữ Hán: phải -> trái
    return ranges[::-1]

def spaced_column_ranges(gray):
    """Tìm cột trên hai nửa trang, dùng khoảng cách giữa các cột dài làm mốc."""
    from scipy.ndimage import gaussian_filter1d
    from scipy.signal import find_peaks

    h, w = gray.shape
    body = gray[round(.08*h):round(.96*h)]
    # Lấy tương phản cục bộ để giấy ố và viền đen không đặt ngưỡng cho cả trang.
    window = max(3, round(.035*w) | 1)
    ink = cv2.adaptiveThreshold(body, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv2.THRESH_BINARY_INV, window, 14)
    _, labels, stats, _ = cv2.connectedComponentsWithStats(ink)
    _, _, widths, heights, areas = stats.T
    strokes = ((widths >= .004*w) & (widths <= .055*w) &
               (heights >= .004*h) & (heights <= .085*h) &
               (areas >= .000018*w*h))
    strokes[0] = False
    mask = strokes[labels]
    profile = gaussian_filter1d(mask.sum(axis=0).astype(float),
                                sigma=max(1, .006*w))
    centers = []
    for left, right in [(0, w//2), (w//2, w)]:
        side = profile[left:right]
        peaks, _ = find_peaks(side, height=.30*side.max(), distance=.033*w)
        anchors = peaks + left
        gaps = np.diff(anchors)
        plausible = gaps[(gaps > .035*w) & (gaps < .065*w)]
        if len(plausible) < 3:
            return None
        pitch = float(np.median(plausible))
        # Nội suy trong khoảng bị khuyết và thử thêm một ô ở hai đầu.
        slots = [anchors[0]-pitch]
        for a, b in zip(anchors, anchors[1:]):
            slots.append(a)
            steps = max(1, round((b-a)/pitch))
            slots.extend(a+(b-a)*j/steps for j in range(1,steps))
        slots.extend([anchors[-1], anchors[-1]+pitch])
        for slot in slots:
            if not left+.36*pitch < slot < right-.36*pitch:
                continue
            radius = round(.20*pitch)
            lo, hi = round(slot)-radius, round(slot)+radius+1
            center = lo+int(np.argmax(profile[lo:hi]))
            # Cực đại nằm sát mép cửa sổ thường là các chấm của lề trang.
            # Vẫn giữ vị trí lệch nếu cụm mực đủ đậm như một chữ.
            at_search_edge = abs(center-slot) >= .16*pitch
            # Một ô chỉ được nhận khi có cụm nét cỡ chữ. Bụi rời và gáy dài
            # đã bị lọc khỏi profile; ô trống không tự sinh cột.
            half_char = round(.31*pitch)
            row_ink = mask[:, center-half_char:center+half_char+1].sum(axis=1)
            glyph_mass = np.convolve(row_ink, np.ones(round(.48*pitch)), mode="valid").max()
            min_mass = .09 if at_search_edge else .05
            if glyph_mass >= min_mass*pitch**2:
                centers.append((center,pitch))
    ranges = []
    for i,(center,pitch) in enumerate(centers):
        half = .48*pitch
        lo = max(round(center-half), (centers[i-1][0]+center)//2 if i else 0)
        hi = min(round(center+half), (centers[i+1][0]+center)//2 if i+1<len(centers) else w)
        ranges.append((lo,hi))
    return ranges[::-1]


def extract_columns_final(
    gray_img,
    cap,
    pad=PAD_FOR_OCR
):
    h, w = gray_img.shape

    ranges = detect_column_ranges(gray_img, cap)

    columns = []

    for start_x, end_x in ranges:

        x1 = max(0, start_x - pad)
        x2 = min(w, end_x + pad)

        if x2 <= x1:
            continue

        col_img = gray_img[:, x1:x2]

        columns.append(col_img)

    return columns



def deskew_mask(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    h, w = gray.shape
    s = min(1.0, MAX_SIDE / max(h, w))
    if s < 1:
        gray = cv2.resize(gray, (max(1, round(w*s)), max(1, round(h*s))),
                          interpolation=cv2.INTER_AREA)
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return gray, mask


def spine_angle(gray):
    h, w = gray.shape
    roi = gray[:, int(w*.40):int(w*.60)]
    if roi.size == 0:
        return None
    edges = cv2.Canny(roi, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/3600, 80,
                            minLineLength=max(1, int(h*.35)),
                            maxLineGap=max(1, int(h*.04)))
    if lines is None:
        return None
    candidates = []
    for x1, y1, x2, y2 in lines.reshape(-1, 4):
        if y2 < y1:
            x1, y1, x2, y2 = x2, y2, x1, y1
        if y2 == y1:
            continue
        a = np.degrees(np.arctan2(float(x2-x1), float(y2-y1)))
        if abs(a) <= MAX_ANGLE:
            candidates.append((float(a), float(np.hypot(x2-x1, y2-y1))))
    if not candidates:
        return None
    candidates.sort(key=lambda v: v[1], reverse=True)
    return float(np.median([a for a, _ in candidates[:7]]))


def text_regions(mask):
    h, w = mask.shape
    # Remove page edges, gutter and horizontal bands from scoring only.
    # Score independent left/right text regions to reduce spine influence.
    regions = []
    for lo, hi in [(0.08, 0.45), (0.55, 0.92)]:
        part = mask[int(h*.10):int(h*.90), int(w*lo):int(w*hi)].copy()
        if part.size == 0:
            continue
        # Remove long connected border strokes if present.
        n, labels, stats, _ = cv2.connectedComponentsWithStats(part, connectivity=8)
        ph, pw = part.shape
        bad = np.zeros(n, dtype=bool)
        for i in range(1, n):
            x, y, cw, ch, area = stats[i]
            bad[i] = (ch > .45*ph) or (cw > .75*pw)
        part[bad[labels]] = 0
        density = np.count_nonzero(part)/part.size
        if .005 < density < .65:
            # Constant canvas with enough room for any tested shear.
            pad = int(np.ceil(ph*np.tan(np.deg2rad(MAX_ANGLE))/2))+3
            regions.append(cv2.copyMakeBorder(part, 0, 0, pad, pad,
                                              cv2.BORDER_CONSTANT, value=0))
    return regions


def projection_score(regions, angle):
    scores = []
    t = np.tan(np.deg2rad(angle))
    for region in regions:
        h, w = region.shape
        # Horizontal shear approximates small rotation for vertical alignment.
        M = np.array([[1, t, -t*(h-1)/2], [0, 1, 0]], dtype=np.float64)
        corrected = cv2.warpAffine(region, M, (w,h), flags=cv2.INTER_LINEAR,
                                   borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        p = corrected.sum(axis=0, dtype=np.float64)
        p = cv2.GaussianBlur(p.reshape(1,-1), (0,0), 1.5).ravel()
        scores.append(float(np.var(p)/(np.mean(p)**2+1e-8)))
    return float(np.mean(scores)) if scores else 0.0


def detect_auto_rotation(img):
    gray, mask = deskew_mask(img)
    hough = spine_angle(gray)
    regions = text_regions(mask)
    if not regions:
        return 0.0, hough, 0.0, 'insufficient text'
    base = projection_score(regions, 0.0)
    def best(angles):
        angles = sorted(set(round(float(a), 4) for a in angles if abs(a) <= MAX_ANGLE+1e-8))
        return max(((a, projection_score(regions,a)) for a in angles),
                   key=lambda pair: (pair[1], -abs(pair[0])))
    coarse = list(np.arange(-MAX_ANGLE, MAX_ANGLE+1e-8, COARSE_STEP))
    if hough is not None:
        coarse.append(hough)
    a, _ = best(coarse)
    a, score = best(list(np.arange(a-.30, a+.301, FINE_STEP))+[a,0.0])
    gain = (score-base)/max(base,1e-8)
    if abs(a) >= MAX_ANGLE-.01:
        return 0.0, hough, gain*100, 'search boundary; inspect'
    if abs(a)<MIN_ANGLE or gain<MIN_GAIN:
        return 0.0, hough, gain*100, 'keep original'
    return round(a, 2), hough, gain*100, 'auto'

## C. Lọc cột biên và phát hiện trường hợp mất/gộp cột — KHÔNG xóa nếu chưa chắc

In [4]:
def border_features(col_img):
    """Đặc trưng hình học dùng để quyết định cột ngoài cùng có phải viền.

    Chỉ phân tích crop trong RAM; không xóa nét hay thay đổi ảnh đưa OCR.
    """
    if col_img is None or col_img.size == 0:
        return {'score': 0.0, 'tall': 0.0, 'large': 0.0,
                'dense_rows': 0.0, 'ink': 0.0, 'text_components': 0}
    gray = (cv2.cvtColor(col_img, cv2.COLOR_BGR2GRAY)
            if col_img.ndim == 3 else col_img)
    h, w = gray.shape
    # Bỏ 10% đầu và cuối để ít bị ảnh hưởng bởi viền ngang mép ảnh.
    body = gray[max(0,int(h*.10)):max(1,int(h*.90)), :]
    if body.size == 0:
        body = gray
    # Đặc trưng tính ở độ phân giải thấp hơn cho nhanh; không thay đổi ảnh OCR.
    if body.shape[0] > 950:
        scale = 950.0 / body.shape[0]
        body = cv2.resize(body, (max(1,round(body.shape[1]*scale)),950),
                          interpolation=cv2.INTER_AREA)
    _, binary = cv2.threshold(body, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    bh, bw = binary.shape
    if bh < 2 or bw < 2:
        return {'score': 0.0, 'tall': 0.0, 'large': 0.0,
                'dense_rows': 0.0, 'ink': 0.0, 'text_components': 0}
    ink = float(np.mean(binary > 0))
    row_density = np.mean(binary > 0, axis=1)
    dense_rows = float(np.mean(row_density >= 0.45))
    n, _, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    tall, large, text_components = 0.0, 0.0, 0
    for x, y, cw, ch, area in stats[1:]:
        if ch >= .58*bh and area >= .025*bh*bw:
            tall = max(tall, float(ch / bh))
            large = max(large, float(area/(bh*bw)))
        if 3 <= cw <= .85*bw and 3 <= ch <= .09*bh and area >= 6:
            text_components += 1
    # Thang điểm conservative; cần review trước khi dùng toàn bộ.
    score = (2.0*tall + 5.0*large + 1.5*dense_rows
             + 1.0*max(0.0, ink-0.20) - min(text_components, 40)*.015)
    return {'score': round(float(score),4), 'tall': round(tall,4),
            'large': round(large,4), 'dense_rows': round(dense_rows,4),
            'ink': round(ink,4), 'text_components': int(text_components)}



def choose_border_columns(columns, cap, page_no):
    n = len(columns)
    indexed = list(enumerate(columns, start=1))
    edges = {i: border_features(columns[i-1]) for i in sorted({1,n})} if n else {}
    proposed = [i for i, f in edges.items() if f['score'] >= BORDER_MIN_SCORE]
    remove_pair = n > EXPECTED_TEXT_COLUMNS and len(proposed) == 2
    removed = proposed.copy() if remove_pair else []
    selected = [(i, col) for i, col in indexed if i not in removed]
    if len(selected) != EXPECTED_TEXT_COLUMNS:
        status = f'REVIEW_{len(selected)}_kept'
    elif proposed and not remove_pair:
        status = 'REVIEW_edge_candidate'
    else:
        status = 'OK'
    if remove_pair:
        status += '_removed_two_edges'
    info = dict(cap=cap, page=int(page_no), detected=n, kept=len(selected),
                removed=removed, proposed=proposed, status=status,
                edge_features={str(i): f for i, f in edges.items()})
    return selected, info


def contact_sheet(columns, info, cap, page, angle):
    """1 JPEG nhỏ / trang, không tạo tệp ảnh trong /kaggle/working."""
    n = len(columns)
    margin, header, footer, gap = 10, 38, 6, 5
    tile_w = THUMB_WIDTH
    width = margin*2 + max(1,n)*(tile_w+gap)-gap
    height = header+THUMB_HEIGHT+footer
    canvas = Image.new('RGB',(width,height),'white')
    draw = ImageDraw.Draw(canvas)
    title=f'{cap} P{page:03d} | {angle:+.2f}° | {n} regions → {info["kept"]} OCR | {info["status"]}'
    draw.text((8,4), title, fill=(0,0,0))
    proposed=set(info.get('proposed',[])); removed=set(info.get('removed',[]))
    for j,gray in enumerate(columns):
        h,w=gray.shape[:2]
        crop=Image.fromarray(gray).convert('L').resize((tile_w,THUMB_HEIGHT),Image.Resampling.BILINEAR).convert('RGB')
        x=margin+j*(tile_w+gap)
        canvas.paste(crop,(x,header))
        color=(205,25,25) if j+1 in removed else ((230,125,10) if j+1 in proposed else (50,85,120))
        draw.rectangle((x,header,x+tile_w-1,header+THUMB_HEIGHT-1),outline=color,width=2 if j+1 in proposed else 1)
        draw.text((x+2,header-17),f'{j+1:02d}',fill=color)
    bio=BytesIO();canvas.save(bio,format='JPEG',quality=PREVIEW_JPEG_QUALITY,optimize=True)
    display(IPyImage(data=bio.getvalue()))


def render_page(doc,page_no):
    pix=doc[page_no-1].get_pixmap(dpi=DPI,colorspace=pymupdf.csRGB,alpha=False)
    rgb=np.frombuffer(pix.samples,dtype=np.uint8).reshape(pix.height,pix.width,3)
    return cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)


def analyze_page(img,cap,page_no):
    angle, hough, gain, skew_status = 0.,None,0.,'not_applied'
    if cap in DESKEW_CAPS:
        crop=img[:int(img.shape[0]*CROP_HEIGHT_RATIO),:]
        angle,hough,gain,skew_status=detect_auto_rotation(crop)
        if USE_MANUAL_OVERRIDE:
            angle=PAGE_ROTATIONS.get((cap,page_no),angle)
            skew_status='manual' if (cap,page_no) in PAGE_ROTATIONS else skew_status
    gray=preprocess_page(img,rotate_deg=angle)
    use_spacing = cap in SPACED_COLUMN_CAPS
    ranges = spaced_column_ranges(gray) if use_spacing else None
    if ranges:
        cols = [gray[:, lo:hi] for lo, hi in ranges]
    else:
        cols = extract_columns_final(gray, cap=cap, pad=PAD_FOR_OCR)
    selected, info = choose_border_columns(cols, cap, page_no)
    if ranges:
        info['ranges_rtl'] = ranges
        info['detector'] = 'spacing'
    elif use_spacing:
        info['detector'] = 'projection_fallback'
        info['status'] = 'REVIEW_spacing_fallback_' + info['status']
    else:
        info['detector'] = 'projection'
    info.update(angle=angle,hough=hough,gain=gain,deskew_status=skew_status)
    return selected,cols,info

In [5]:
def choose_border_columns(columns, cap, page_no):
    n = len(columns)

    # Số ảnh đúng theo thứ tự hiển thị trên preview, bắt đầu từ 01.
    if n == 17:
        candidates = [17]
    elif n == 18:
        candidates = [1, 18]
    else:
        candidates = []

    edge_features = {}
    removed = []

    for col_no in candidates:
        features = border_features(columns[col_no - 1])
        edge_features[str(col_no)] = features

        if features["score"] >= BORDER_MIN_SCORE:
            removed.append(col_no)

    # Giữ nguyên số cột gốc để đối chiếu preview với kết quả OCR.
    selected = [
        (col_no, image)
        for col_no, image in enumerate(columns, start=1)
        if col_no not in removed
    ]

    kept = len(selected)
    status = "OK" if kept == EXPECTED_TEXT_COLUMNS else f"REVIEW_{kept}_kept"
    if removed:
        suffix = "_".join(f"{col_no:02d}" for col_no in removed)
        status += f"_removed_{suffix}"

    info = {
        "cap": cap,
        "page": int(page_no),
        "detected": n,
        "kept": kept,
        "removed": removed,
        "proposed": removed.copy(),
        "status": status,
        "edge_features": edge_features,
    }
    return selected, info

## E. Chạy theo lô, theo dõi cảnh báo, lưu kết quả từng trang

In [6]:
SAMPLE_PAGES = {2, 6, 48}

def unwrap_result_json(data):
    if isinstance(data,dict):
        if 'rec_texts' in data:return data
        for key in ('res','data','result'):
            if isinstance(data.get(key),dict) and 'rec_texts' in data[key]:return data[key]
    return None


def extract_segments(data):
    texts=data.get('rec_texts') or []
    scores=data.get('rec_scores')
    polys=data.get('rec_polys')
    if polys is None or len(polys)==0:polys=data.get('dt_polys')
    scores=[] if scores is None else scores
    polys=[] if polys is None else polys
    rows=[]
    for i,t in enumerate(texts):
        if not str(t).strip():continue
        score=float(scores[i]) if i<len(scores) and scores[i] is not None else None
        poly=polys[i] if i<len(polys) else None
        top=min(float(pt[1]) for pt in poly) if poly is not None and len(poly) else float(i)
        rows.append((top,i,str(t),score))
    rows.sort(key=lambda x:(x[0],x[1]))
    return [(t,conf) for _,_,t,conf in rows]


def json_default(obj):
    if isinstance(obj,np.ndarray):return obj.tolist()
    if isinstance(obj,np.generic):return obj.item()
    if isinstance(obj,Path):return str(obj)
    raise TypeError(type(obj))


def ocr_one_page(selected, cap, page, info):
    setup_ocr()
    prefix=f'{cap.replace("_","")}-{page:03d}'
    path=OUTPUT_ROOT / cap / 'HAN' / 'pages'
    path.mkdir(parents=True,exist_ok=True)
    data_rows=[]; all_text=[]; seq=0
    save_raw = page in SAMPLE_PAGES  # CHỈ 02, 06, 48 của cả Cap_01 và Cap_02
    for original_col, col_img in selected:
        if save_raw:
            # Giống notebook cũ: truyền đường dẫn ảnh PNG vào PaddleOCR,
            # giữ ảnh cột gốc và dùng save_to_img để vẽ chữ OCR cạnh ảnh.
            name = f'{page:04d}-{original_col:02d}'
            folder = OUTPUT_ROOT / cap / 'HAN' / 'raw' / f'{page:04d}' / name
            folder.mkdir(parents=True, exist_ok=True)
            input_png = folder / f'{name}.png'
            if not cv2.imwrite(str(input_png), col_img):
                raise OSError(f'Cannot save OCR input: {input_png}')
            ocr_input = str(input_png)
        else:
            # Các trang khác giữ nguyên phương thức mới: OCR trong RAM,
            # KHÔNG tạo ảnh cột, raw hay bất kỳ file tạm nào.
            ocr_input = cv2.cvtColor(col_img,cv2.COLOR_GRAY2BGR)

        predictions=ocr_han.predict(ocr_input,
            text_det_limit_side_len=2000,text_det_limit_type='max',
            text_det_box_thresh=.3,text_det_unclip_ratio=1.5)
        parts=[]
        for result_no, r in enumerate(predictions, 1):
            raw=r.json if hasattr(r,'json') else dict(r)
            if callable(raw):raw=raw()
            if isinstance(raw,str):raw=json.loads(raw)
            part=unwrap_result_json(raw)
            if part is None:raise ValueError(f'Missing rec_texts for {prefix} col {original_col}')
            segments = extract_segments(part)
            if save_raw:
                # save_to_img như bản cũ: ảnh có chú thích văn bản bên cạnh.
                stem = name if result_no == 1 else f'{name}-{result_no}'
                r.save_to_img(str(folder / f'{stem}_ocr_res_img.png'))
            parts.extend(segments)
        if save_raw:
            # JSON đúng định dạng của notebook cũ: chỉ rec_texts/rec_scores.
            # Không xuất toàn bộ JSON PaddleOCR như bản mới.
            old_raw_json = {'rec_texts': [t for t, _ in parts],
                            'rec_scores': [score for _, score in parts]}
            (folder / f'{name}_res.json').write_text(
                json.dumps(old_raw_json, ensure_ascii=False, indent=2), encoding='utf-8')
            (folder / 'result.txt').write_text(
                ''.join(t for t, _ in parts), encoding='utf-8')
        for text,conf in parts:
            seq+=1; data_rows.append(dict(id=f'{prefix}-{seq:03d}',han=text,confidence=conf,
                                           cap=cap,page=page,segment=seq,column=f'{prefix}-COL{original_col:02d}'))
            all_text.append(text)
    payload=dict(info=info,rows=data_rows,text=''.join(all_text))
    (path / f'{prefix}.json').write_text(json.dumps(payload,ensure_ascii=False,default=json_default),encoding='utf-8')
    (path / f'{prefix}.txt').write_text(payload['text'],encoding='utf-8')
    return seq


def run_all(start=1,count=10,do_ocr=False,show_preview=None,caps=None):
    """Mỗi lô gồm [start..start+count-1] cho cả hai PDF. Không xoá kết quả cũ."""
    if show_preview is None:show_preview=SHOW_PREVIEW
    if count<1 or start<1:raise ValueError('start>=1, count>=1')
    if do_ocr:
        print('Đang khởi tạo PaddleOCR...', flush=True)
        setup_ocr()
    report=[]; t0=time.perf_counter()
    active_caps = CAPS if caps is None else caps
    total = sum(max(0, min(count, page_counts[cap] - start + 1)) for cap in active_caps)
    for cap in active_caps:
        with pymupdf.open(PDF_PATHS[cap]) as doc:
            end=min(start+count-1,len(doc))
            for page in range(start,end+1):
                done = len(report)
                eta = (time.perf_counter()-t0)*(total-done)/done if done else None
                remaining = f'{eta/60:.1f} phút' if eta is not None else 'đang ước tính'
                print(f'[{done}/{total} | {100*done/total:.0f}%] {cap} P{page:03d} | Còn khoảng {remaining}', flush=True)
                try:
                    img=render_page(doc,page)
                    selected,cols,info=analyze_page(img,cap,page)
                    if show_preview:contact_sheet(cols,info,cap,page,info['angle'])
                    if do_ocr:
                        # Chỉ OCR các vùng đã qua bước lọc viền.
                        nseg=ocr_one_page(selected,cap,page,info)
                    else:nseg=None
                    row={k:info[k] for k in ('cap','page','detected','kept','status','angle','gain','removed','proposed','detector')}
                    row['segments']=nseg
                    report.append(row)
                    del img,selected,cols
                except Exception as exc:
                    report.append(dict(cap=cap,page=page,status='ERROR',error=repr(exc)))
                    print(f'ERROR {cap} P{page}: {exc}')
                    # Giữ khả năng chạy các trang còn lại nhưng không ẩn lỗi khỏi summary.
                gc.collect()
    print(f'[{len(report)}/{total}] Đã xử lý hết các trang. Lỗi: {sum(r["status"] == "ERROR" for r in report)}', flush=True)
    df=pd.DataFrame(report)
    if len(df):
        print(df.to_string(index=False), flush=True)
        warn=df[df['status'].fillna('').astype(str).str.startswith(('REVIEW','ERROR'))]
        print(f'Pages: {len(df)} | REVIEW/ERROR: {len(warn)} | Elapsed: {time.perf_counter()-t0:.1f}s')
    return df


def build_exports():
    """Đọc các trang OCR đã lưu (không phụ thuộc state của các cell chạy batch)."""
    rows=[];pages=[];texts={cap:[] for cap in CAPS}
    for cap in CAPS:
        for f in sorted((OUTPUT_ROOT / cap / 'HAN' / 'pages').glob('*.json')):
            try:
                obj=json.loads(f.read_text(encoding='utf-8'))
                d=obj['info']; rr=obj['rows']; rows.extend(rr)
                conf=[r['confidence'] for r in rr if r.get('confidence') is not None]
                pages.append(dict(cap=cap,page=d['page'],detected=d['detected'],kept=d['kept'],
                                  removed=','.join(map(str,d['removed'])),status=d['status'],
                                  segments=len(rr),mean_confidence=sum(conf)/len(conf) if conf else None))
                texts[cap].append((d['page'],obj['text']))
            except Exception as exc: print('BAD PAGE FILE',f,exc)
    if not pages:
        print('Chưa có dữ liệu OCR. Chạy run_all(...,do_ocr=True) trước.');return None
    xlsx=OUTPUT_ROOT/'HAN_OCR_batch.xlsx'
    with pd.ExcelWriter(xlsx,engine='xlsxwriter') as writer:
        pd.DataFrame(rows).to_excel(writer,sheet_name='OCR_chi_tiet',index=False)
        pd.DataFrame(pages).to_excel(writer,sheet_name='Tong_hop_trang',index=False)
    for cap,content in texts.items():
        (OUTPUT_ROOT/f'{cap}_HAN_all_pages.txt').write_text('\n\n'.join(
            f'=== {cap} | PAGE {p:03d} ===\n{txt}' for p,txt in sorted(content)),encoding='utf-8')
    from shutil import make_archive
    archive=make_archive(str(OUTPUT_ROOT),'zip',root_dir=OUTPUT_ROOT)
    print('Excel:',xlsx,'ZIP:',archive,'Pages:',len(pages))
    return xlsx,archive

In [7]:
page_counts = {}
for cap in CAPS:
    with pymupdf.open(PDF_PATHS[cap]) as doc:
        page_counts[cap] = len(doc)
print(page_counts)
report = run_all(count=max(page_counts.values()), do_ocr=True, show_preview=False)
report.to_csv(OUTPUT_ROOT / 'run_report.csv', index=False)
errors = report.loc[report['status'].eq('ERROR')]
if len(errors):
    print(errors.to_string(index=False), flush=True)
if len(report) - len(errors) != sum(page_counts.values()):
    raise RuntimeError('OCR còn trang lỗi. Xem run_report.csv trước khi xuất ZIP.')

{'Cap_01': 53, 'Cap_02': 92}
Đang khởi tạo PaddleOCR...


/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Paddle: 3.2.0 GPU count: 2


W1123 15:30:24.122997  7831 gpu_resources.cc:114] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 13.0, Runtime API Version: 12.6


GPU Conv2D OK


Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


[0/145 | 0%] Cap_01 P001 | Còn khoảng đang ước tính
[1/145 | 1%] Cap_01 P002 | Còn khoảng 20.1 phút
[2/145 | 1%] Cap_01 P003 | Còn khoảng 20.5 phút
[3/145 | 2%] Cap_01 P004 | Còn khoảng 19.8 phút
[4/145 | 3%] Cap_01 P005 | Còn khoảng 19.2 phút
[5/145 | 3%] Cap_01 P006 | Còn khoảng 18.8 phút
[6/145 | 4%] Cap_01 P007 | Còn khoảng 18.8 phút
[7/145 | 5%] Cap_01 P008 | Còn khoảng 18.5 phút
[8/145 | 6%] Cap_01 P009 | Còn khoảng 18.1 phút
[9/145 | 6%] Cap_01 P010 | Còn khoảng 17.7 phút
[10/145 | 7%] Cap_01 P011 | Còn khoảng 17.6 phút
[11/145 | 8%] Cap_01 P012 | Còn khoảng 17.3 phút
[12/145 | 8%] Cap_01 P013 | Còn khoảng 17.1 phút
[13/145 | 9%] Cap_01 P014 | Còn khoảng 16.9 phút
[14/145 | 10%] Cap_01 P015 | Còn khoảng 16.8 phút
[15/145 | 10%] Cap_01 P016 | Còn khoảng 16.6 phút
[16/145 | 11%] Cap_01 P017 | Còn khoảng 16.5 phút
[17/145 | 12%] Cap_01 P018 | Còn khoảng 16.3 phút
[18/145 | 12%] Cap_01 P019 | Còn khoảng 16.2 phút
[19/145 | 13%] Cap_01 P020 | Còn khoảng 16.0 phút
[20/145 | 14%] Cap_0

In [8]:
excel_path, zip_path = build_exports()
from IPython.display import HTML, Javascript
from IPython.display import display
import os


# Kaggle phục vụ tệp trong working qua /files/ trên máy chủ notebook.
from IPython.display import FileLink
from pathlib import Path
from uuid import uuid4
os.chdir('/kaggle/working')
download_id = 'download_' + uuid4().hex
relative_zip = Path(zip_path).relative_to(Path.cwd())
link = FileLink(str(relative_zip), result_html_prefix='', result_html_suffix='')
display(HTML(f'<span id="{download_id}">{link._repr_html_()}</span>'))
display(Javascript(f"""
const link = document.querySelector('#{download_id} a');
if (link) {{
    link.download = '{relative_zip.name}';
    link.click();
}}
"""))


Excel: /kaggle/working/HAN_OCR_full_k4f3ggky/HAN_OCR_batch.xlsx ZIP: /kaggle/working/HAN_OCR_full_k4f3ggky.zip Pages: 145


<IPython.core.display.Javascript object>